In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import json
from collections import Counter

In [3]:
sys.path.insert(0, "../../scripts")

from paths import ALIGNMENT_JSON, BOOK_TEXT, TEST_AUDIO, RECORDING_WAV, CLIPS_DIR, MANIFEST_ALL, MANIFEST_CLEAN
from book_check import check_against_book
from chunks import make_chunks
from grading import grade_chunks
from clips import decode_to_wav, write_clips, write_manifest

In [4]:
words = json.loads(ALIGNMENT_JSON.read_text(encoding="utf-8"))
words = check_against_book(words, BOOK_TEXT.read_text(encoding="utf-8"))

confirmed = sum(word["confirmed"] for word in words)
print(f"Stage 10: {confirmed} of {len(words)} words confirmed by the book ({100 * confirmed / len(words):.1f}%)")

Stage 10: 831 of 875 words confirmed by the book (95.0%)


In [5]:
decode_to_wav(TEST_AUDIO, RECORDING_WAV)
chunks = make_chunks(words, RECORDING_WAV)

durations = [chunk["duration"] for chunk in chunks]
print(f"Stage 11: {len(chunks)} chunks, {min(durations)}-{max(durations)} s, average {sum(durations) / len(durations):.1f} s")

Stage 11: 47 chunks, 7.46-19.24 s, average 12.1 s


In [6]:
chunks = grade_chunks(chunks, words)

print("Stage 12:", dict(Counter(chunk["tier"] for chunk in chunks)))
print()
for chunk in chunks:
    if chunk["tier"] != "clean":
        print(f"{chunk['tier']:7} {chunk['start']:7.2f} s   {'; '.join(chunk['reasons'])}")


Stage 12: {'review': 15, 'clean': 20, 'reject': 12}

review    10.91 s   not confirmed by the book: Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.; cut where the audio is not silent: listen to the edges
review    24.16 s   not confirmed by the book: Qadim oʻtirar ekan.; cut where the audio is not silent: listen to the edges
review    32.46 s   not confirmed by the book: qoʻli gʻamginsiz,
review    69.31 s   not confirmed by the book: vazirdan
review    91.77 s   not confirmed by the book: birortasi podshoh
review   132.88 s   not confirmed by the book: amaldorlarga
reject   167.60 s   unsure word 'bora' (score -4.92); unsure word 'qol,' (score -3.23)
reject   190.91 s   unsure word 'dam' (score -3.65)
review   202.28 s   not confirmed by the book: parini
review   222.96 s   not confirmed by the book: hurliqo,
reject   238.21 s   unsure word 'Men' (score -8.56)
reject   251.44 s   unsure word 'devdan' (score -3.5)


In [7]:
chunks = write_clips(chunks, RECORDING_WAV, CLIPS_DIR, "test")

write_manifest(chunks, MANIFEST_ALL, {"clean", "review", "reject"})
write_manifest(chunks, MANIFEST_CLEAN, {"clean"})

for tier in ["clean", "review", "reject"]:
    seconds = sum(chunk["duration"] for chunk in chunks if chunk["tier"] == tier)
    count = sum(chunk["tier"] == tier for chunk in chunks)
    print(f"{tier:7} {count:3} clips  {seconds / 60:5.2f} min")
print()
print("Clips in:", CLIPS_DIR)
print("Training manifest:", MANIFEST_CLEAN)


clean    20 clips   3.91 min
review   15 clips   2.97 min
reject   12 clips   2.59 min

Clips in: /home/intern/book_audio_alignment/data/test/chunks/wavs
Training manifest: /home/intern/book_audio_alignment/data/test/manifest_clean.jsonl


In [8]:
from IPython.display import Audio, display

for chunk in chunks:
    if chunk["tier"] != "clean":
        print(f"{chunk['tier']}  {chunk['start']:.2f} s  {'; '.join(chunk['reasons'])}")
        print("   ", chunk["text"])
        display(Audio(chunk["audio_filepath"]))

review  10.91 s  not confirmed by the book: Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.; cut where the audio is not silent: listen to the edges
    Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.


review  24.16 s  not confirmed by the book: Qadim oʻtirar ekan.; cut where the audio is not silent: listen to the edges
    Qadim zamonda bir podsho oʻtgan ekan, U taxtda doim gʻamgin oʻtirar ekan.


review  32.46 s  not confirmed by the book: qoʻli gʻamginsiz,
    Kunlarning birida oʻng qoʻli vaziri undan, Podshohim, nimagadir mudom gʻamginsiz, boisi nima? deb soʻrabdi.


review  69.31 s  not confirmed by the book: vazirdan
    shu bargdan olib kelishni buyuring, podsho vazirdan bu maslahatni eshitib xursand boʻlibdi va shaharga jarchi qoʻydiribdi,


review  91.77 s  not confirmed by the book: birortasi podshoh
    Bu ovozani eshitib, shoyad olib kelolsak, oliy hazratning in’omiga muyassar boʻlardik, deydiganlar koʻpayib qolibdi-yu, lekin birortasi ham yurak yutib podshoh huzuriga kelmabdi.


review  132.88 s  not confirmed by the book: amaldorlarga
    baxtiyor hukmdorga shart qoʻyibdi, Shartim bor, kambagʻallarga yordam bersangiz, podsho bu shartni eshitib, vazir-u amaldorlarga maslahat solibdi,


... (17 outputs left out)

reject  500.47 s  unsure word 'azamat,' (score -6.14)
    Baxtiyor yana uzoq yoʻl bosib, Gulnora parining qoshiga kelibdi, Yigitni koʻrib xursand boʻlgan qiz, Senga qoyil boʻldim, azamat, devlarning otasini oʻldirib ajab qilibsan, sara qilibsan, oʻgʻillari otalarining qotilini izlab motam tutyaptilar,


review  529.56 s  not confirmed by the book: tovoze
    podsho Baxtiyorning kelayotganini eshitib, amaldorlari bilan yoʻliga peshvoz chiqibdi, yigit daraxti baxtning bargini tovoze bilan podshoga beribdi,


reject  558.76 s  unsure word 'Hamma' (score -3.4)
    oradan bir yil oʻtgach, mamlakat hukmdori oʻgʻil koʻrdi degan xushxabar tarqalibdi. Hamma shod-u xurram boʻlib, murod-maqsadiga yetibdi. Podsho uchun daraxti baxt


reject  572.25 s  unsure word 'Bolalar' (score -8.22)
    izlagan Baxtiyor ham Gulnora pari bilan baxtli hayot kechiribdi. Bolalar adabiyoti orzu-niyatlaringiz kaliti.
